<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №6: Синтез цифровых фильтров (КИХ и БИХ)

## Часть 1. Синтез КИХ-фильтров методом окон

### Задание 1.1. Проектирование ФНЧ с разными окнами
Спроектируйте КИХ-ФНЧ с частотой среза 200 Гц (частота дискретизации 1000 Гц), длина фильтра 51 отсчёт. Используйте окна:
- прямоугольное,
- Ханна,
- Хемминга,
- Блэкмана.

Постройте на одном графике АЧХ (в дБ) всех четырёх фильтров. Сравните:
- крутизну среза,
- уровень пульсаций в полосе пропускания и заграждения,
- ширину переходной полосы.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 1000
fc = 200
numtaps = 51
windows = ['boxcar', 'hann', 'hamming', 'blackman']

plt.figure(figsize=(11, 6))
metrics = {}

for w in windows:
    b = signal.firwin(numtaps, fc, window=w, fs=fs)
    f, h = signal.freqz(b, worN=8192, fs=fs)
    H = np.abs(h)
    HdB = 20 * np.log10(np.maximum(H, 1e-12))

    plt.plot(f, HdB, label=w)

    stop_att = -np.max(HdB[f >= 250])
    pass_ripple = np.max(HdB[f <= 150]) - np.min(HdB[f <= 150])

    idx_09 = np.where(H <= 0.9)[0][0]
    idx_01 = np.where(H <= 0.1)[0][0]
    trans_width = f[idx_01] - f[idx_09]

    metrics[w] = {
        'stop_att_db': stop_att,
        'pass_ripple_db': pass_ripple,
        'trans_width_hz': trans_width,
    }

plt.title('FIR LPF (fc=200 Hz, N=51): different windows')
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.xlim(0, fs / 2)
plt.ylim(-120, 5)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

print('Metrics summary:')
for w in windows:
    m = metrics[w]
    print(
        f"{w:8s} | stopband attenuation: {m['stop_att_db']:.2f} dB | "
        f"passband ripple: {m['pass_ripple_db']:.3f} dB | "
        f"transition width: {m['trans_width_hz']:.2f} Hz"
    )


**Вопросы:** Какое окно обеспечивает наилучшее подавление в полосе заграждения? Какое – самую крутую переходную полосу? Как это связано с формой окна?

Окно Блэкмана обеспечивает наилучшее подавление в полосе заграждения, т.к. у него самый низкий уровень боковых лепестков. Прямоугольное окно имеет самую крутую переходную полосу. Чем резче обрывается окно по краям, тем уже главный лепесток спектра окна, что дает высокую крутизну среза. Резкие края порождают  пульсации, поэтому подавление в полосе заграждения слабое. Чем плавнее окно спадает к краям, тем слабее пульсации



### Задание 1.2. Влияние длины фильтра
Для окна Хемминга спроектируйте ФНЧ с fc=200 Гц, fs=1000 Гц, с длинами 21, 51, 101. Постройте АЧХ на одном графике.


In [95]:
fs = 1000
fc = 200
lengths = [21, 51, 101]

plt.figure(figsize=(11, 6))

for L in lengths:
    b = signal.firwin(L, fc, window='hamming', fs=fs)
    f, h = signal.freqz(b, worN=8192, fs=fs)
    HdB = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    plt.plot(f, HdB, label=f'N={L}')

plt.title('Hamming FIR LPF: effect of filter length (fc=200 Hz)')
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.xlim(0, fs / 2)
plt.ylim(-120, 5)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


**Вопрос:** Как увеличение длины влияет на крутизну среза и на уровень боковых лепестков?

С увеличением длины фильтра крутизна среза  увеличивается. Максимальный уровень боковых лепестков практически не меняется, так как он определяется типом используемого окна. Сами лепестки становятся уже и их количество увеличивается




### Задание 1.3. Синтез ФВЧ и полосового фильтра методом окон
Используя тот же оконный метод, спроектируйте:
- ФВЧ с fc=200 Гц (длина 51, окно Хемминга);
- полосовой фильтр с полосой пропускания 200–300 Гц (длина 51, окно Хемминга).

Постройте АЧХ обоих фильтров.


In [94]:
fs = 1000
numtaps = 51

b_high = signal.firwin(numtaps, 200, pass_zero=False, window='hamming', fs=fs)
b_band = signal.firwin(numtaps, [200, 300], pass_zero='bandpass', window='hamming', fs=fs)

plt.figure(figsize=(11, 6))
for b, name in [(b_high, 'HPF (fc=200 Hz)'), (b_band, 'Band-pass (200-300 Hz)')]:
    f, h = signal.freqz(b, worN=8192, fs=fs)
    HdB = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    plt.plot(f, HdB, label=name)

plt.title('HPF and BPF FIR responses (Hamming, N=51)')
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.xlim(0, fs / 2)
plt.ylim(-120, 5)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


Подайте на спроектированные фильтры сигнал: сумма синусоид 100, 250 и 350 Гц (амплитуды 1, 0.8, 0.6). Постройте спектры до и после фильтрации (в логарифмическом масштабе).

In [93]:
fs = 1000
t = np.arange(0, 1, 1 / fs)

x = 1.0 * np.sin(2 * np.pi * 100 * t) + 0.8 * np.sin(2 * np.pi * 250 * t) + 0.6 * np.sin(2 * np.pi * 350 * t)

y_high = signal.lfilter(b_high, [1], x)
y_band = signal.lfilter(b_band, [1], x)

def spectrum(sig, fs):
    N = len(sig)
    S = np.fft.rfft(sig)
    f = np.fft.rfftfreq(N, 1 / fs)
    A = 2 * np.abs(S) / N
    return f, A

f0, A0 = spectrum(x, fs)
f1, A1 = spectrum(y_high, fs)
f2, A2 = spectrum(y_band, fs)

plt.figure(figsize=(11, 8))

plt.subplot(2, 1, 1)
plt.plot(f0, A0, label='Input')
plt.plot(f1, A1, label='After HPF')
plt.plot(f2, A2, label='After BPF')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude')
plt.title('Spectra (linear scale)')
plt.grid(True, alpha=0.4)
plt.legend()

plt.subplot(2, 1, 2)
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input')
plt.semilogy(f1, np.maximum(A1, 1e-12), label='After HPF')
plt.semilogy(f2, np.maximum(A2, 1e-12), label='After BPF')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude (log)')
plt.title('Spectra (log scale)')
plt.grid(True, alpha=0.4)
plt.legend()

plt.tight_layout()
plt.show()


**Вопрос:** Объясните полученные результаты.

В исходном сигнале три компоненты: 100Гц, 250Гц и 350Гц. ФВЧ с частотой среза 200Гц  подавил низкочастотную составляющую(100Гц) и пропустил 250 и 350Гц. Полосовой фильтр с полосой 200–300Гц подавил частоты 100Гц и 350Гц и оставил только компоненту 250 Гц. Различия в итоговых амплитудах подавленных гармоник  зависят от глубины подавления фильтра на данной частоте согласно его АЧХ



## Часть 2. Равноволновой синтез КИХ-фильтров (Паркса–МакКлеллана)

### Задание 2.1. Проектирование оптимального ФНЧ
Используя `signal.remez`, спроектируйте ФНЧ с параметрами:
- частота дискретизации 1000 Гц,
- полоса пропускания 0–150 Гц,
- полоса заграждения 250–500 Гц,
- длина фильтра 21.

Постройте АЧХ (в линейном и логарифмическом масштабах) и сравните её с АЧХ КИХ-фильтра, полученного методом окон (окно Хемминга, та же длина, fc=200 Гц).

In [92]:
fs = 1000
numtaps = 21

b_remez = signal.remez(numtaps, [0, 150, 250, 500], [1, 0], fs=fs)
b_hamm = signal.firwin(numtaps, 200, window='hamming', fs=fs)

f_r, h_r = signal.freqz(b_remez, worN=8192, fs=fs)
f_w, h_w = signal.freqz(b_hamm, worN=8192, fs=fs)

Hr = np.abs(h_r)
Hw = np.abs(h_w)

plt.figure(figsize=(11, 8))

plt.subplot(2, 1, 1)
plt.plot(f_r, Hr, label='remez (equiripple)')
plt.plot(f_w, Hw, label='firwin + hamming')
plt.xlim(0, 500)
plt.ylim(0, 1.2)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude')
plt.title('Magnitude response (linear), N=21')
plt.grid(True, alpha=0.4)
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(f_r, 20 * np.log10(np.maximum(Hr, 1e-12)), label='remez (equiripple)')
plt.plot(f_w, 20 * np.log10(np.maximum(Hw, 1e-12)), label='firwin + hamming')
plt.xlim(0, 500)
plt.ylim(-100, 5)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.title('Magnitude response (log), N=21')
plt.grid(True, alpha=0.4)
plt.legend()

plt.tight_layout()
plt.show()



**Вопросы:** Какой фильтр имеет более крутой срез? Каковы пульсации в полосе пропускания?

Более крутой срез имеет фильтр, спроектированный равноволновым методом(remez). Он оптимизируется для обеспечения заданного перехода между полосами. У равноволнового фильтра в полосе пропускания и заграждения наблюдаются пульсации равной амплитуды. У фильтра Хемминга АЧХ в полосе пропускания монотонно спадает

### Задание 2.2. Зависимость от ширины переходной полосы

Поменяйте верхнюю частоту полосы пропускания и нижнюю частоту полосы заграждения так, чтобы их среднее оставалось равным 200 Гц. Постройте АЧХ получившегося фильтра. Подберите ширину переходной области (разности нижней частоты полосы заграждения и верхней частоты полосы пропускания) так, чтобы уровень подавления в полосе заграждения у равноволнового фильтра совпал с оконным фильтром Хемминга.


In [ ]:
fs = 1000
numtaps = 21

# Reference: Hamming window FIR
b_ref = signal.firwin(numtaps, 200, window='hamming', fs=fs)
f_ref, h_ref = signal.freqz(b_ref, worN=8192, fs=fs)
att_ref = -np.max(20 * np.log10(np.maximum(np.abs(h_ref[f_ref >= 250]), 1e-12)))

width_candidates = list(range(20, 241, 10))
results = []

for width in width_candidates:
    fp = 200 - width / 2
    fsb = 200 + width / 2
    if fp <= 0 or fsb >= fs / 2:
        continue

    b = signal.remez(numtaps, [0, fp, fsb, fs / 2], [1, 0], fs=fs)
    f, h = signal.freqz(b, worN=8192, fs=fs)

    att = -np.max(20 * np.log10(np.maximum(np.abs(h[f >= fsb]), 1e-12)))
    results.append((width, fp, fsb, att, abs(att - att_ref), b, f, h))

best = min(results, key=lambda x: x[4])
best_width, best_fp, best_fsb, best_att, _, best_b, best_f, best_h = best

plt.figure(figsize=(11, 6))
plt.plot(best_f, 20 * np.log10(np.maximum(np.abs(best_h), 1e-12)), label=f'remez, transition={best_width} Hz')
plt.plot(f_ref, 20 * np.log10(np.maximum(np.abs(h_ref), 1e-12)), '--', label='firwin+hamming (reference)')
plt.xlim(0, 500)
plt.ylim(-100, 5)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.title('Transition width tuning for matched stopband attenuation')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

print(f'Reference attenuation (hamming): {att_ref:.2f} dB')
print(f'Best remez transition width: {best_width} Hz')
print(f'Band edges: pass <= {best_fp:.1f} Hz, stop >= {best_fsb:.1f} Hz')
print(f'Remez attenuation: {best_att:.2f} dB')


**Вопрос:** При какой ширине переходной области уровень подавления в полосе заграждения у равноволнового фильтра совпал с оконным фильтром Хемминга?

При N=21 подавление в полосе заграждения у remez достигает уровня окна Хемминга при ширине переходной области около 160

### Задание 2.3. Управление весами
Для фильтра из задачи 2.1 измените весовые коэффициенты: задайте `weight=[1, 10]` (увеличить вес для полосы заграждения). Постройте новую АЧХ и сравните с предыдущей.


In [91]:
fs = 1000
numtaps = 21
bands = [0, 150, 250, 500]
desired = [1, 0]

b_remez_eq = signal.remez(numtaps, bands, desired, weight=[1, 1], fs=fs)
b_remez_w = signal.remez(numtaps, bands, desired, weight=[1, 10], fs=fs)

f, h_eq = signal.freqz(b_remez_eq, worN=8192, fs=fs)
_, h_w = signal.freqz(b_remez_w, worN=8192, fs=fs)

plt.figure(figsize=(11, 6))
plt.plot(f, 20 * np.log10(np.maximum(np.abs(h_eq), 1e-12)), label='weight=[1,1]')
plt.plot(f, 20 * np.log10(np.maximum(np.abs(h_w), 1e-12)), label='weight=[1,10]')
plt.xlim(0, 500)
plt.ylim(-120, 5)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.title('Weight impact in remez design')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

def metrics(h):
    HdB = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    stop_att = -np.max(HdB[f >= 250])
    pass_ripple = np.max(HdB[f <= 150]) - np.min(HdB[f <= 150])
    return stop_att, pass_ripple

att_eq, rip_eq = metrics(h_eq)
att_w, rip_w = metrics(h_w)

print(f'weight=[1,1]  -> stop: {att_eq:.2f} dB, pass ripple: {rip_eq:.3f} dB')
print(f'weight=[1,10] -> stop: {att_w:.2f} dB, pass ripple: {rip_w:.3f} dB')


**Вопрос:** Как изменилось подавление в полосе заграждения и пульсации в полосе пропускания?

Установка веса 10 для полосы заграждения привело к тому что алгоритм сосредоточился на минимизации ошибки, подавление в полосе заграждения значительно улучшилось. Но это произошло за счёт ухудшения в другой полосе, пульсации в полосе пропускания увеличились

### Задание 2.4. Сравнение КИХ-фильтров на реальном сигнале
Сгенерируйте сигнал: смесь синусоид 50 Гц, 120 Гц, 220 Гц (амплитуды 1, 0.7, 0.3) + белый шум (дисперсия 0.1), fs=1000 Гц. Пропустите этот сигнал через:
- КИХ-ФНЧ спроектированный методом окон (окно Хемминга, длина 51, fc=150 Гц);
- КИХ-ФНЧ спроектированный методом `remez` (длина 51, полоса пропускания 0–100 Гц, полоса заграждения 200–500 Гц).

Постройте спектры исходного и отфильтрованных сигналов в линейном и логарифмическом масштабе.


In [90]:
np.random.seed(42)

fs = 1000
t = np.arange(0, 2, 1 / fs)

x = (
    1.0 * np.sin(2 * np.pi * 50 * t)
    + 0.7 * np.sin(2 * np.pi * 120 * t)
    + 0.3 * np.sin(2 * np.pi * 220 * t)
    + np.random.normal(0, np.sqrt(0.1), len(t))
)

b_win = signal.firwin(51, 150, window='hamming', fs=fs)
b_remez_51 = signal.remez(51, [0, 100, 200, 500], [1, 0], fs=fs)

y_win = signal.lfilter(b_win, [1], x)
y_remez = signal.lfilter(b_remez_51, [1], x)

def spec(sig):
    N = len(sig)
    S = np.fft.rfft(sig)
    f = np.fft.rfftfreq(N, 1 / fs)
    A = 2 * np.abs(S) / N
    return f, A

f0, A0 = spec(x)
f1, A1 = spec(y_win)
f2, A2 = spec(y_remez)

plt.figure(figsize=(12, 9))

plt.subplot(2, 1, 1)
plt.plot(f0, A0, label='Input')
plt.plot(f1, A1, label='After FIR (Hamming)')
plt.plot(f2, A2, label='After FIR (remez)')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude')
plt.title('Spectra (linear)')
plt.grid(True, alpha=0.4)
plt.legend()

plt.subplot(2, 1, 2)
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input')
plt.semilogy(f1, np.maximum(A1, 1e-12), label='After FIR (Hamming)')
plt.semilogy(f2, np.maximum(A2, 1e-12), label='After FIR (remez)')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude (log)')
plt.title('Spectra (log)')
plt.grid(True, alpha=0.4)
plt.legend()

plt.tight_layout()
plt.show()

def amp_at(sig, f0):
    f, A = spec(sig)
    idx = np.argmin(np.abs(f - f0))
    return A[idx]

a120_in = amp_at(x, 120)
a120_win = amp_at(y_win, 120)
a120_remez = amp_at(y_remez, 120)
a220_in = amp_at(x, 220)
a220_win = amp_at(y_win, 220)
a220_remez = amp_at(y_remez, 220)

print(f'Amplitude at 120 Hz: input={a120_in:.4f}, Hamming={a120_win:.4f}, remez={a120_remez:.4f}')
print(f'Amplitude at 220 Hz: input={a220_in:.4f}, Hamming={a220_win:.4f}, remez={a220_remez:.4f}')


**Вопросы:** Какой фильтр лучше подавил 220 Гц и шум? Какой лучше сохранил 120 Гц? На сколько отличаются амплитуды 120 Гц сигнала от исходных для каждого из фильтров?


220 Гц и шум лучше подавил фильтр remez. На логарифмическом графике видно, что в полосе заграждения, выше 200 Гц, уровень полки шума и пика 220 Гц у remez опускается ниже

120 Гц лучше сохранил фильтр Hamming. У него частота среза задана как 150Гц, поэтому частота 120Гц полностью попадает в его полосу пропускания и практически не ослабляется. У фильтра remez полоса пропускания гарантирована только до 100Гц, а область от 100 до 200Гц является переходной полосой, где АЧХ падает, из-за чего частота 120 Гц подвергается искажению

Для Hamming: Амплитуда частоты 120 Гц практически не изменилась и неотличима от исходного уровня

Для Remez: Из-за попадания в переходную зону амплитуда 120 Гц существенно просела

## Часть 3. Синтез БИХ-фильтров

### Задание 3.1. Баттерворт, Чебышев, эллиптический – сравнение АЧХ
Спроектируйте ФНЧ с частотой среза 200 Гц (fs=1000 Гц) следующих типов (порядок 4):
- Баттерворт (`butter`);
- Чебышев I с пульсациями 1 дБ (`cheby1`);
- Чебышев II с затуханием 40 дБ в полосе заграждения (`cheby2`);
- Эллиптический с пульсациями 1 дБ и затуханием 40 дБ (`ellip`).

Постройте АЧХ всех фильтров на одном графике (в дБ). Сравните:
- крутизну среза,
- пульсации в полосе пропускания и заграждения.

Постройте и сравните фазовые характеристики (ФЧХ) и групповые задержки.


In [89]:
fs = 1000
fc = 200
order = 4

filters = {
    'Butterworth': signal.butter(order, fc, btype='low', fs=fs),
    'Chebyshev I (1 dB)': signal.cheby1(order, 1, fc, btype='low', fs=fs),
    'Chebyshev II (40 dB)': signal.cheby2(order, 40, fc, btype='low', fs=fs),
    'Elliptic (1/40 dB)': signal.ellip(order, 1, 40, fc, btype='low', fs=fs),
}

plt.figure(figsize=(11, 6))
for name, (b, a) in filters.items():
    f, h = signal.freqz(b, a, worN=8192, fs=fs)
    plt.plot(f, 20 * np.log10(np.maximum(np.abs(h), 1e-12)), label=name)

plt.title('IIR filters, order 4: magnitude responses')
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.xlim(0, 500)
plt.ylim(-100, 5)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

plt.figure(figsize=(11, 6))
for name, (b, a) in filters.items():
    f, h = signal.freqz(b, a, worN=2048, fs=fs)
    phase = np.unwrap(np.angle(h))
    plt.plot(f, phase, label=name)

plt.title('IIR filters, order 4: phase responses')
plt.xlabel('Frequency, Hz')
plt.ylabel('Phase, rad')
plt.xlim(0, 500)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

plt.figure(figsize=(11, 6))
for name, (b, a) in filters.items():
    fg, gd = signal.group_delay((b, a), w=1024, fs=fs)
    plt.plot(fg, gd, label=name)

plt.title('IIR filters, order 4: group delay')
plt.xlabel('Frequency, Hz')
plt.ylabel('Delay, samples')
plt.xlim(0, 500)
plt.ylim(bottom=0)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


**Вопросы:** Какой фильтр даёт самый крутой спад? Какой имеет наименьшие фазовые искажения в полосе пропускания?


Эллиптический фильтр даёт самый крутой спад. Наименьшие фазовые искажения в полосе пропускания имеет фильтр Баттерворта, у него ФЧХ близка к линейной, а групповая задержка более плоская в зоне пропускания

### Задание 3.2. Влияние порядка на характеристики БИХ-фильтра
Для фильтра Баттерворта с fc=200 Гц, fs=1000 Гц возьмите порядки 2, 4, 6. Постройте АЧХ и групповую задержку.


In [88]:
fs = 1000
fc = 200
orders = [2, 4, 6]

plt.figure(figsize=(11, 6))

for n in orders:
    b, a = signal.butter(n, fc, btype='low', fs=fs)
    f, h = signal.freqz(b, a, worN=8192, fs=fs)
    plt.plot(f, 20 * np.log10(np.maximum(np.abs(h), 1e-12)), label=f'order={n}')

plt.title('Butterworth: effect of order on magnitude response')
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.xlim(0, 500)
plt.ylim(-100, 5)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

plt.figure(figsize=(11, 6))
for n in orders:
    b, a = signal.butter(n, fc, btype='low', fs=fs)
    fg, gd = signal.group_delay((b, a), w=1024, fs=fs)
    plt.plot(fg, gd, label=f'order={n}')

plt.title('Butterworth: effect of order on group delay')
plt.xlabel('Frequency, Hz')
plt.ylabel('Delay, samples')
plt.xlim(0, 500)
plt.ylim(bottom=0)
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


**Вопросы:** Как увеличение порядка влияет на крутизну среза и на групповую задержку в полосе пропускания?

При увеличении порядка переходная полоса сужается. При этом увеличивается групповая задержка вблизи частоты среза, что значит рост фазовых искажений сигнала

### Задание 3.3. Преобразование типа БИХ-фильтра (ФНЧ → ФВЧ, полосовой)
Спроектируйте Баттерворта 4-го порядка:
- ФНЧ с fc=200 Гц;
- ФВЧ с fc=200 Гц (используйте `btype='high'`);
- полосовой с полосой 200–300 Гц.

Подайте на них сигнал: сумма синусоид 100, 250, 400 Гц. Постройте спектры после фильтрации.


In [87]:
fs = 1000

b_lpf, a_lpf = signal.butter(4, 200, btype='low', fs=fs)
b_hpf, a_hpf = signal.butter(4, 200, btype='high', fs=fs)
b_bpf, a_bpf = signal.butter(4, [200, 300], btype='bandpass', fs=fs)

t = np.arange(0, 1, 1 / fs)
x = np.sin(2 * np.pi * 100 * t) + np.sin(2 * np.pi * 250 * t) + np.sin(2 * np.pi * 400 * t)

y_lpf = signal.lfilter(b_lpf, a_lpf, x)
y_hpf = signal.lfilter(b_hpf, a_hpf, x)
y_bpf = signal.lfilter(b_bpf, a_bpf, x)

def spec(sig):
    N = len(sig)
    f = np.fft.rfftfreq(N, 1 / fs)
    A = 2 * np.abs(np.fft.rfft(sig)) / N
    return f, A

f0, A0 = spec(x)
fl, Al = spec(y_lpf)
fh, Ah = spec(y_hpf)
fb, Ab = spec(y_bpf)

plt.figure(figsize=(12, 9))

plt.subplot(3, 1, 1)
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input', alpha=0.5)
plt.semilogy(fl, np.maximum(Al, 1e-12), label='After LPF')
plt.xlim(0, 500)
plt.ylabel('Amplitude (log)')
plt.title('LPF output spectrum')
plt.grid(True, alpha=0.4)
plt.legend()

plt.subplot(3, 1, 2)
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input', alpha=0.5)
plt.semilogy(fh, np.maximum(Ah, 1e-12), label='After HPF')
plt.xlim(0, 500)
plt.ylabel('Amplitude (log)')
plt.title('HPF output spectrum')
plt.grid(True, alpha=0.4)
plt.legend()

plt.subplot(3, 1, 3)
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input', alpha=0.5)
plt.semilogy(fb, np.maximum(Ab, 1e-12), label='After BPF')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude (log)')
plt.title('BPF output spectrum')
plt.grid(True, alpha=0.4)
plt.legend()

plt.tight_layout()
plt.show()


**Вопрос:** Какие составляющие подавлены, а какие пропущены в каждом случае?

ФНЧ пропустил 100Гц, подавил 250 и 400Гц

ФВЧ подавил 100Гц, пропустил 250, 400Гц

Полосовой пропустил 250Гц, подавил 100 и 400Гц,

### Задание 3.4. Применение БИХ-фильтра к зашумлённому сигналу
Сгенерируйте сигнал: синусоида 50 Гц + белый шум с дисперсией 0.2, fs=1000 Гц, длительность 1 с. Спроектируйте эллиптический ФНЧ с fc=100 Гц (порядок 6, пульсации 1 дБ, затухание 40 дБ). Примените фильтр с помощью `lfilter` и `filtfilt`. Постройте на одном графике исходный и отфильтрованные сигналы (временные области, первые 0.2 с), а также их спектры. Сравните задержку и подавление шума.


In [ ]:
np.random.seed(42)

fs = 1000
t = np.arange(0, 1, 1 / fs)

x = np.sin(2 * np.pi * 50 * t) + np.random.normal(0, np.sqrt(0.2), len(t))

b_ell, a_ell = signal.ellip(6, 1, 40, 100, btype='low', fs=fs)

y_lf = signal.lfilter(b_ell, a_ell, x)
y_ff = signal.filtfilt(b_ell, a_ell, x)

# Time domain (first 0.2 s)
idx = int(0.2 * fs)

plt.figure(figsize=(12, 5))
plt.plot(t[:idx], x[:idx], label='Input', alpha=0.6)
plt.plot(t[:idx], y_lf[:idx], label='lfilter')
plt.plot(t[:idx], y_ff[:idx], label='filtfilt')
plt.xlabel('Time, s')
plt.ylabel('Amplitude')
plt.title('Time-domain signals (first 0.2 s)')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

# Spectra

def spec(sig):
    N = len(sig)
    f = np.fft.rfftfreq(N, 1 / fs)
    A = 2 * np.abs(np.fft.rfft(sig)) / N
    return f, A

f0, A0 = spec(x)
f1, A1 = spec(y_lf)
f2, A2 = spec(y_ff)

plt.figure(figsize=(12, 5))
plt.semilogy(f0, np.maximum(A0, 1e-12), label='Input', alpha=0.6)
plt.semilogy(f1, np.maximum(A1, 1e-12), label='lfilter')
plt.semilogy(f2, np.maximum(A2, 1e-12), label='filtfilt')
plt.xlim(0, 500)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude (log)')
plt.title('Spectra')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

# Delay estimation by cross-correlation peak
corr_l = np.correlate(y_lf, x, mode='full')
corr_f = np.correlate(y_ff, x, mode='full')
lags = np.arange(-len(x) + 1, len(x))
lag_l = lags[np.argmax(corr_l)]
lag_f = lags[np.argmax(corr_f)]

print(f'Estimated delay (lfilter): {lag_l} samples')
print(f'Estimated delay (filtfilt): {lag_f} samples')


**Вопрос:** Почему `filtfilt` даёт нулевой фазовый сдвиг?

Функция filtfilt устраняет фазовый сдвиг потому что она применяет фильтр дважды: сначала в прямом направлении, а затем в обратном, что математически компенсирует задержку, введенную при первом проходе.

## Часть 4. Сравнение КИХ и БИХ фильтров одинакового порядка

### Задание 4.1. Сравнение характеристик
Спроектируйте:
- КИХ-ФНЧ методом окон (окно Хемминга, длина 51, fc=200 Гц);
- БИХ-ФНЧ Баттерворта 4-го порядка (fc=200 Гц).

Постройте для них на одном графике:
- отдельно АЧХ (в дБ),
- отдельно групповую задержку.


In [86]:
fs = 1000
fc = 200

b_fir = signal.firwin(51, fc, window='hamming', fs=fs)
b_iir, a_iir = signal.butter(4, fc, btype='low', fs=fs)

f_fir, h_fir = signal.freqz(b_fir, worN=8192, fs=fs)
f_iir, h_iir = signal.freqz(b_iir, a_iir, worN=8192, fs=fs)

plt.figure(figsize=(11, 6))
plt.plot(f_fir, 20 * np.log10(np.maximum(np.abs(h_fir), 1e-12)), label='FIR (Hamming, N=51)')
plt.plot(f_iir, 20 * np.log10(np.maximum(np.abs(h_iir), 1e-12)), label='IIR Butterworth, order=4')
plt.xlim(0, 500)
plt.ylim(-120, 5)
plt.xlabel('Frequency, Hz')
plt.ylabel('Amplitude, dB')
plt.title('FIR vs IIR magnitude responses')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

fg_fir, gd_fir = signal.group_delay((b_fir, [1]), w=1024, fs=fs)
fg_iir, gd_iir = signal.group_delay((b_iir, a_iir), w=1024, fs=fs)

plt.figure(figsize=(11, 6))
plt.plot(fg_fir, gd_fir, label='FIR (Hamming, N=51)')
plt.plot(fg_iir, gd_iir, label='IIR Butterworth, order=4')
plt.xlim(0, 500)
plt.ylim(bottom=0)
plt.xlabel('Frequency, Hz')
plt.ylabel('Delay, samples')
plt.title('Group delay comparison')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()

# Butterworth order with similar slope at 250 Hz
att_fir_250 = -20 * np.log10(np.maximum(np.abs(h_fir[np.argmin(np.abs(f_fir - 250))]), 1e-12))
best = None

for n in range(2, 31):
    b, a = signal.butter(n, fc, btype='low', fs=fs)
    f, h = signal.freqz(b, a, worN=8192, fs=fs)
    att_250 = -20 * np.log10(np.maximum(np.abs(h[np.argmin(np.abs(f - 250))]), 1e-12))
    err = abs(att_250 - att_fir_250)
    if best is None or err < best[1]:
        best = (n, err, att_250)

print(f'FIR attenuation @250 Hz: {att_fir_250:.2f} dB')
print(f'Closest Butterworth order: n={best[0]}, attenuation @250 Hz={best[2]:.2f} dB')


**Вопросы:** Какой фильтр имеет более крутой срез? Какой имеет постоянную групповую задержку? Какой вносит меньшие фазовые искажения? Какой требует меньше вычислений? Попробуйте увеличить порядок фильтра Баттерворта. При каком порядке крутизна среза его АЧХ становится сравнимой с КИХ-фильтром? Растут или уменьшаются при этом фазовые искажения?


При частоте дискретизации 1000Гц КИХ-фильтр имеет более крутой срез АЧХ в переходной зоне. КИХ-фильтр имеет постоянную групповую задержку. КИХ-фильтр вносит нулевые фазовые искажения, так как постоянная групповая задержка означает линейную ФЧХ, БИХ-фильтр имеет нелинейную фазу. БИХ-фильтр требует меньше вычислений, для расчета одного выходного отсчета ему требуется 9 операций, а КИХ-фильтру 51 операция.

Крутизна среза БИХ-фильтра становится сравнимой с КИХ-фильтром при повышении порядка до 8-10.
С увеличением порядка БИХ-фильтру будет становиться круче. Всплеск групповой задержки вблизи частоты среза становится острее, что ведет к искажению формы сигналов

### Задание 4.2. Применение к реальному сигналу
Сгенерируйте сигнал: короткий прямоугольный импульс длительностью 10 мс, а также высокочастотная синусоидальная помеха 400 Гц. Частоту дискретизации возьмите fs=10000 Гц.

Примените оба фильтра (КИХ и БИХ из предыдущего пункта). Постройте исходный и отфильтрованные сигналы.


In [85]:
fs = 10000
t = np.arange(0, 0.2, 1 / fs)

# Rectangular pulse, duration 10 ms
pulse = np.where(t < 0.01, 1.0, 0.0)

# High-frequency interference
interf = 0.4 * np.sin(2 * np.pi * 400 * t)

x = pulse + interf

# Filters from 4.1, recomputed for fs=10000
b_fir = signal.firwin(51, 200, window='hamming', fs=fs)
b_iir, a_iir = signal.butter(4, 200, btype='low', fs=fs)

y_fir = signal.lfilter(b_fir, [1], x)
y_iir = signal.lfilter(b_iir, a_iir, x)

plt.figure(figsize=(12, 6))
plt.plot(t, x, label='Input', alpha=0.7)
plt.plot(t, y_fir, label='After FIR')
plt.plot(t, y_iir, label='After IIR (Butterworth)')
plt.xlabel('Time, s')
plt.ylabel('Amplitude')
plt.title('Pulse + 400 Hz interference: input and filtered signals')
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


**Вопросы:** Какой фильтр лучше сохранил форму импульсов? Какой лучше подавил помеху?

КИХ-фильтр лучше сохранил форму импульсов. Оба фильтра сглаживают углы прямоугольника из-за низкой частоты среза. КИХ-фильтр сохраняет симметрию импульса благодаря постоянной групповой задержке. БИХ-фильтр из-за нелинейной фазы искажает форму, спад затягивается

БИХ-фильтр лучше подавил помеху. На графиках спектров видно, что на частоте 400 Гц пик помехи после БИХ-фильтра падает вниз. У КИХ-фильтра на этой частоте подавление слабое